# Weight Symmetry Analysis

Detect symmetry and structure in model weights: QK/OV symmetry,
MLP transpose symmetry, cross-head diversity, and embed-unembed alignment.

## Why This Matters

Weight symmetry examines the structure and statistics of model parameters. The structure of weights constrains what computations a component can perform, and spectral analysis can reveal functional specialization, redundancy, and low-rank structure.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.weight_symmetry_analysis import (
    qk_symmetry, ov_symmetry,
    mlp_weight_symmetry, cross_head_symmetry,
    embed_unembed_symmetry,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print('Model ready')

## QK Symmetry

Symmetric QK = content-based matching; antisymmetric = positional structure.

In [ ]:
for layer in range(4):
    for head in range(4):
        result = qk_symmetry(model, layer=layer, head=head)
        sym = 'SYM' if result['is_symmetric'] else 'asym'
        print(f"  L{layer}H{head}: sym={result['symmetric_fraction']:.3f}, "
              f"anti={result['antisymmetric_fraction']:.3f} [{sym}]")

## OV Symmetry

Symmetric OV circuit suggests the head copies information bidirectionally.

In [ ]:
for layer in range(4):
    for head in range(4):
        result = ov_symmetry(model, layer=layer, head=head)
        sym = 'SYM' if result['is_symmetric'] else 'asym'
        print(f"  L{layer}H{head}: sym={result['symmetric_fraction']:.3f}, "
              f"anti={result['antisymmetric_fraction']:.3f} [{sym}]")

## MLP Weight Symmetry

Is W_out approximately the transpose of W_in (autoencoder-like structure)?

In [ ]:
for layer in range(4):
    result = mlp_weight_symmetry(model, layer=layer)
    flag = ' [TRANSPOSE]' if result['is_approximately_transpose'] else ''
    print(f"  Layer {layer}: cos(W_in^T, W_out)={result['transpose_cosine']:.4f}{flag}")

## Cross-Head Symmetry

Are heads specialized (diverse) or redundant (similar)?

In [ ]:
for layer in range(4):
    result = cross_head_symmetry(model, layer=layer)
    flag = ' [DIVERSE]' if result['heads_are_diverse'] else ''
    print(f"  Layer {layer}: QK_sim={result['mean_qk_similarity']:.4f}, "
          f"OV_sim={result['mean_ov_similarity']:.4f}{flag}")

## Embed-Unembed Symmetry

Is W_U approximately W_E^T (weight tying)?

In [ ]:
result = embed_unembed_symmetry(model)
print(f"Global cosine(W_E^T, W_U): {result['global_cosine']:.4f}")
print(f"Mean per-token cosine: {result['mean_per_token_cosine']:.4f}")
print(f"Weight tied: {result['is_weight_tied']}")
print(f"W_E norm: {result['W_E_norm']:.4f}, W_U norm: {result['W_U_norm']:.4f}")